In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [9]:
PROJECT_ROOT = Path('.').resolve() # Assuming you run this in project root

print("🔍 Loading data for LSTM Sequence generation...")
df = pd.read_parquet("C:/Users/Sam Garcia/PycharmProjects/macro_alpha/data/processed/train_ready_features.parquet")

# We only want the LSTM to look at pure price action and volatility 
# (XGBoost handles the macro data)
features = ['daily_return', 'volatility_20d', 'RSI_14']
target = 'target_5d_up'

🔍 Loading data for LSTM Sequence generation...


In [4]:
# Neural Networks are highly sensitive to scale, so we must normalize
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[features])

# 🛠️ Create 10-Day Sequential Windows
SEQUENCE_LENGTH = 10

X_seq, y_seq = [], []
for i in range(len(scaled_features) - SEQUENCE_LENGTH):
    X_seq.append(scaled_features[i:(i + SEQUENCE_LENGTH)])
    y_seq.append(df[target].iloc[i + SEQUENCE_LENGTH])

X_seq = torch.FloatTensor(np.array(X_seq))
y_seq = torch.FloatTensor(np.array(y_seq)).unsqueeze(1)

print(f"Created {len(X_seq)} sequential samples of {SEQUENCE_LENGTH} days each.")

Created 3832 sequential samples of 10 days each.


In [5]:
# 🧠 Define the LSTM Architecture
class MarketLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(MarketLSTM, self).__init__()
        # The LSTM layer memory cells
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        # The final decision layer
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Pass through LSTM
        lstm_out, _ = self.lstm(x)
        # Grab the output from the very last day of the 10-day sequence
        last_day_out = lstm_out[:, -1, :]
        # Pass through the linear layer and squash between 0 and 1
        out = self.fc(last_day_out)
        return self.sigmoid(out)

In [6]:
# Initialize Model
model = MarketLSTM(input_size=len(features), hidden_size=32, num_layers=2)
criterion = nn.BCELoss() # Binary Cross Entropy for 0/1 prediction
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print("\n🚀 Training Deep Learning LSTM (100 Epochs)...")
# For this prototype, we'll do a simple Train/Test split (first 80% train, last 20% test)
split = int(len(X_seq) * 0.8)
X_train, y_train = X_seq[:split], y_seq[:split]
X_test, y_test = X_seq[split:], y_seq[split:]

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    # Backward pass and optimize
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


🚀 Training Deep Learning LSTM (100 Epochs)...
Epoch [20/100], Loss: 0.6716
Epoch [40/100], Loss: 0.6696
Epoch [60/100], Loss: 0.6622
Epoch [80/100], Loss: 0.6263
Epoch [100/100], Loss: 0.5998


In [7]:
# 📊 Evaluate the LSTM
model.eval()
with torch.no_grad():
    test_predictions = model(X_test)
    # Use our strict 55% confidence threshold
    test_preds_binary = (test_predictions > 0.55).float().numpy()
    
    prec = precision_score(y_test.numpy(), test_preds_binary)
    print("\n=========================================")
    print(f"🤖 LSTM Standalone Precision: {prec:.2%}")
    print("=========================================")


🤖 LSTM Standalone Precision: 62.07%
